# 8장 — 안전장치들: 정규화 🔗

7장의 암기 곡선에 원점 목줄(weight decay)을 채웁니다 — 같은 데이터, 같은 차수 9.

**이 노트북의 목표**
1. 암기 곡선의 정체: 폭주하는 계수(‖w‖ 수백) — 크기 벌점 λ‖w‖²로 묶으면 규칙만 남는다
2. λ의 U자: 너무 느슨하면 암기, 너무 빡빡하면 과소적합 — λ도 검증으로 고르는 다이얼
3. GD에 목줄을 채우면 갈라짐(7장) 자체가 사라진다 — 조기 종료 없이도 안전

In [ ]:
# 🤖 AI 튜터 — 막히면 물어보세요 (로컬 Ollama, 없어도 학습 지장 없음)
from tutor import chat
chat('ch8')

## 1. 실험 ① — 원점 목줄: λ 스위프

7장과 같은 데이터(시드 21/22), 같은 차수 9. 벌점 붙은 최소제곱(릿지)은 닫힌 형태로 풀립니다:
$(F^\top F/n + \lambda I)\,w = F^\top y/n$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

TRUE = lambda x: np.sin(2.5 * x)
SIGMA = 0.25

def make_data(n, seed):
    rng = np.random.default_rng(seed)
    x = np.clip(np.linspace(-1, 1, n) + rng.uniform(-0.04, 0.04, n), -1, 1)
    return x, TRUE(x) + SIGMA * rng.standard_normal(n)

xt, yt = make_data(10, 21)   # 7장과 동일
xv, yv = make_data(30, 22)

power = lambda x: np.stack([x ** k for k in range(10)], axis=1)
Ft, Fv = power(xt), power(xv)
n = len(yt)

def ridge(lam):
    w = np.linalg.solve(Ft.T @ Ft / n + lam * np.eye(10), Ft.T @ yt / n)
    return w, np.mean((Ft @ w - yt) ** 2), np.mean((Fv @ w - yv) ** 2), np.linalg.norm(w)

lams = [0, 1e-4, 1e-3, 1e-2, 0.1, 1, 10]
print('   λ        훈련      검증     ‖w‖ (계수 크기)')
results = {}
for lam in lams:
    w, tr, va, nw = ridge(lam)
    results[lam] = (w, tr, va, nw)
    print(f'  {lam:7g}  {tr:.5f}  {va:.4f}  {nw:8.2f}')

# 곡선 3종: 목줄 없음 / 적당 / 과잉
gx = np.linspace(-1, 1, 300)
gF = power(gx)
plt.figure(figsize=(7.5, 3.4))
plt.plot(gx, TRUE(gx), 'k--', alpha=0.4, label='숨은 규칙')
for lam, c in [(0, 'tab:red'), (1e-3, 'tab:green'), (10, 'tab:blue')]:
    plt.plot(gx, np.clip(gF @ results[lam][0], -2.5, 2.5), color=c, label=f'λ = {lam:g}')
plt.scatter(xt, yt, c='indigo', zorder=5, s=30)
plt.ylim(-2.5, 2.5); plt.legend(fontsize=8); plt.title('목줄 없음(암기) / 적당(규칙) / 과잉(짓눌림)')
plt.tight_layout(); plt.show()

norms = [results[l][3] for l in lams]
vals = {l: results[l][2] for l in lams}
assert norms[0] > 100, '목줄 없는 암기 곡선은 계수가 수백까지 폭주'
assert all(a > b for a, b in zip(norms, norms[1:])), 'λ가 셀수록 ‖w‖ 단조 감소 — 목줄이 조여든다'
assert vals[1e-3] < vals[0] / 4, '적당한 목줄: 검증 오차 4분의 1 이하 (암기 소멸)'
assert vals[10] > 4 * vals[1e-3], '과잉 목줄: 규칙까지 짓눌려 과소적합'
assert results[10][1] > 0.5, '과소적합은 훈련 오차도 나쁘다 (7장 U자의 왼쪽과 같은 증상)'
print('\n✔ 확인: 암기에는 큰 계수가 필요하다 — 벌점으로 묶으면 규칙만 남고, 과하면 규칙까지 짓눌린다 (λ의 U자)')

## 2. 실험 ② — 목줄 경주: 갈라짐이 사라진다

7장 실험 ②의 GD(체비쇼프 기저, 3,000스텝)를 λ=0 vs λ=0.03으로 두 번 —
조기 종료 없이 완주했을 때 검증 손실은?

In [ ]:
def cheb(x, deg=9):
    T = [np.ones_like(x), x]
    for k in range(2, deg + 1):
        T.append(2 * x * T[-1] - T[-2])
    return np.stack(T[: deg + 1], axis=1)

Ct, Cv = cheb(xt), cheb(xv)

def gd(lam, steps=3000, lr=0.05):
    w = np.zeros(10)
    trs, vas = [], []
    for t in range(steps):
        r = Ct @ w - yt
        w -= lr * (2 * Ct.T @ r / n + 2 * lam * w)   # 벌점 기울기 2λw 추가
        trs.append(np.mean(r ** 2))
        vas.append(np.mean((Cv @ w - yv) ** 2))
    return np.array(trs), np.array(vas)

tr0, va0 = gd(0)       # 목줄 없음 (7장 그대로)
trL, vaL = gd(0.03)    # 목줄 채움
k0 = int(va0.argmin())

plt.figure(figsize=(7.5, 3.4))
plt.plot(tr0, color='indigo', alpha=0.4, label='훈련 손실 (λ=0)')
plt.plot(va0, color='tab:red', label='검증, 목줄 없음')
plt.plot(vaL, color='tab:green', lw=2, label='검증, 목줄 λ=0.03')
plt.axvline(k0, color='green', ls=':', alpha=0.5)
plt.ylim(0, 0.6); plt.xlabel('스텝'); plt.legend(); plt.grid(alpha=0.3)
plt.title('목줄을 채우면 갈라짐 자체가 생기지 않는다')
plt.tight_layout(); plt.show()

print(f'목줄 없음: 검증 최저 {va0[k0]:.4f}@{k0} → 마지막 {va0[-1]:.4f} ({va0[-1]/va0[k0]:.1f}배 되오름)')
print(f'목줄 0.03: 마지막 검증 {vaL[-1]:.4f} (최저의 {vaL[-1]/vaL.min():.2f}배 — 갈라짐 없음)')
print(f'목줄 값:   마지막 훈련 오차 {tr0[-1]:.5f} → {trL[-1]:.5f} (조금 손해 — 잃은 건 암기 점수뿐)')

assert va0[-1] > 3 * va0[k0], '목줄 없으면: 최저 후 3배 이상 되오름 (7장의 갈라짐)'
assert vaL[-1] < 1.1 * va0[k0], '목줄 채우면: 완주하고도 목줄 없는 쪽의 최저 수준 이하'
assert vaL[-1] < 1.1 * vaL.min(), '갈라짐 자체가 없다 — 마지막이 곧 최저 수준'
assert trL[-1] > tr0[-1] * 10, '목줄 값: 훈련 오차는 손해 본다 (그것이 암기 점수)'
print('\n✔ 확인: 멈출 타이밍을 몰라도 안전 — 벌점이 노이즈 암기용 계수를 계속 깎아낸다')

## 3. 실험 ③ — 이름의 정체: '감쇠' 인자 확인

In [ ]:
# w ← (1 − 2·lr·λ)·w − lr·∇L — 데이터 항이 없으면(∇L=0) 순수 감쇠만 남는다
lr, lam = 0.1, 0.05
factor = 1 - 2 * lr * lam
print(f'감쇠 인자 = 1 − 2×{lr}×{lam} = {factor}')

w = 1.0
for t in range(100):
    w *= factor          # 데이터가 우기지 않는 계수의 운명
print(f'데이터가 안 우기는 계수: 100스텝 후 {w:.4f} (0으로 스르르)')

assert abs(factor - 0.99) < 1e-12, 'StepQuiz 값: 감쇠 인자 0.99'
assert w < 0.4, '방치된 계수는 원점으로 끌려간다'
print('\n✔ 확인: weight decay = 매 스텝 (1−2·lr·λ)를 곱하는 것 — 이름 그대로 감쇠')

## 4. 관찰 정리 + 🤖 AI 연결

- **weight decay**: 손실 + λ‖w‖²(1부 노름) — 암기 곡선엔 폭주하는 계수가 필요하니 크기를
  비싸게. 갱신식으로 풀면 매 스텝 (1−2·lr·λ)를 곱하는 감쇠. 설정: `weight_decay: 0.01`.
- **λ의 U자**: 느슨하면 암기가 남고 빡빡하면 규칙까지 짓눌림(과소적합) — λ도 검증으로 고른다.
- **갈라짐 소멸**: 목줄을 채우면 조기 종료 없이 완주해도 검증이 최저 수준 — 훈련 오차의
  손해분은 정확히 '암기 점수'라 남는 장사.
- **3형제**: 원점 목줄(weight decay)·무작위 결근(드롭아웃 — 담합 암기 방지)·제때
  하차(조기 종료). 본질은 하나 — 암기를 어렵게.
- **KL 목줄과의 대구(6부 6장)**: 손실 + λ×(거리 벌점) — 기준점이 원본 모델(KL)이냐
  원점(decay)이냐. AdamW = Adam + 분리된 감쇠(`optimizer: adamw`의 W).

**다음 챕터**: 9장 — 훈련 일지 판독법. 다이얼 상자가 다 찼으니 이제 계기판 읽기 —
손실 곡선의 4대 증상(스파이크·평원·계단·벌어짐) 진단과 처방입니다.